# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli from microsoft/DeBERTa-v3-base

Model page: https://huggingface.co/MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli

Entailment is calculated in both directions (forward probs (Arg1->Arg2) and backward probs (Arg2->Arg1)), then the label is decided with the following criteria:
   
Bidirectional decision:
  - Rephrase: high entailment both ways, low contradiction
  - Attack: contradiction high either way
  - Support: entailment high at least one way, and not clearly neutral/contradictory
  - No Relationship: otherwise

# No keywords

In [1]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships No Keywords"

model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 50                 
MODEL_TAG = "deberta"      
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

# figure out label 
id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
label2id = {v: int(k) for k, v in id2label.items()}

print(f'DeBERTa v3 base mnli labels {label2id}')

IDX_ENT = label2id["entailment"]
IDX_CON = label2id["contradiction"]
IDX_NEU = label2id["neutral"]

# Heuristic thresholds 
ENT_THR       = 0.50   
CONTR_THR     = 0.50   
REPHRASE_THR  = 0.75  
MAX_NEUTRAL   = 0.70   

# Margins: “X wins by at least this much”
SUPPORT_MARGIN = 0.10  # p_ent - p_con must exceed this (either direction)
ATTACK_MARGIN  = 0.10  # p_con - p_ent must exceed this (either direction)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2025-08-25 13:39:14.239681: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756129154.570518      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756129154.664623      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

DeBERTa v3 base mnli labels {'entailment': 0, 'neutral': 1, 'contradiction': 2}


In [2]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2459, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 2459 (delta 9), reused 6 (delta 5), pack-reused 2444 (from 2)
Receiving objects: 100% (2459/2459), 97.50 MiB | 17.15 MiB/s, done.
Resolving deltas: 100% (2065/2065), done.
Updating files: 100% (1061/1061), done.


In [3]:
def preprocess_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

@torch.no_grad()
def nli_probs(pairs, max_length: int):
    if not pairs:
        return np.zeros((0,3)), np.zeros((0,3))

    a1 = [preprocess_text(x[0]) for x in pairs]
    a2 = [preprocess_text(x[1]) for x in pairs]

    # forward: premise=a1, hypothesis=a2
    enc_f = tokenizer(a1, a2, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_f = model(**enc_f).logits
    prob_f = torch.softmax(logits_f, dim=-1).detach().cpu().numpy()

    # backward: premise=a2, hypothesis=a1
    enc_b = tokenizer(a2, a1, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_b = model(**enc_b).logits
    prob_b = torch.softmax(logits_b, dim=-1).detach().cpu().numpy()

    return prob_f, prob_b

def decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b):
    # 1) Hard guard: if both sides look very neutral, say NR
    if max(p_neu_f, p_neu_b) >= MAX_NEUTRAL:
        return "No Relationship"

    # 2) Rephrase: bi-directional entailment and low contradiction
    if min(p_ent_f, p_ent_b) >= REPHRASE_THR and max(p_con_f, p_con_b) <= (1 - REPHRASE_THR):
        return "Rephrase"

    # 3) Attack: contradiction wins with margin OR strong contradiction
    if (
        (p_con_f - p_ent_f) >= ATTACK_MARGIN or
        (p_con_b - p_ent_b) >= ATTACK_MARGIN or
        p_con_f >= CONTR_THR or
        p_con_b >= CONTR_THR
    ):
        return "Attack"

    # 4) Support: entailment wins with margin OR clears lowered threshold
    if (
        (p_ent_f - p_con_f) >= SUPPORT_MARGIN or
        (p_ent_b - p_con_b) >= SUPPORT_MARGIN or
        p_ent_f >= ENT_THR or
        p_ent_b >= ENT_THR
    ):
        return "Support"

    # 5) Fallback
    return "No Relationship"


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_max_length(input_dir, prefix_substring, safety_limit=512):
    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = preprocess_text(str(a) if pd.notna(a) else "")
            b = preprocess_text(str(b) if pd.notna(b) else "")
            ids = tokenizer(a, b, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(ids))
    if max_len == 0:
        max_len = 128
    return min(max_len, safety_limit)

def classify_relationships_deberta(input_dir, prefix_substring, model_tag=MODEL_TAG):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    MAX_LENGTH = compute_max_length(input_dir, prefix_substring, safety_limit=512)
    print(f"Using MAX_LENGTH = {MAX_LENGTH}")

    rel_col = f"rel_{model_tag}"

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]

            pairs = list(zip(chunk["SDGarg1"].astype(str).tolist(),
                             chunk["SDGarg2"].astype(str).tolist()))
            try:
                prob_f, prob_b = nli_probs(pairs, max_length=MAX_LENGTH)
                labels = []
                for i in range(len(pairs)):
                    pf = prob_f[i]; pb = prob_b[i]
                    p_ent_f, p_neu_f, p_con_f = pf[IDX_ENT], pf[IDX_NEU], pf[IDX_CON]
                    p_ent_b, p_neu_b, p_con_b = pb[IDX_ENT], pb[IDX_NEU], pb[IDX_CON]
                    lab = decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b)
                    labels.append(lab)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(pairs)

            df.loc[chunk.index, rel_col] = labels

            # first 5 examples
            for (a1, a2), lab in zip(pairs, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Sample predictions (first 5):\n")
                    for _a1, _a2, _lab in first_examples:
                        print(f"- Arg1: {_a1}\n- Arg2: {_a2}\n  Label: {_lab}\n")
                    first_examples.append((a1, a2, lab))

            total_done += len(pairs)
            if total_done % 100 < BATCH_SIZE:
                print(f"  Progress: {total_done}/{n} relations classified...")

        # Final label validation 
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
    return df


## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 160

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: According to major international studies, few teenagers can differentiate between a fact and an opinion.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: As the world’s nations prepare to meet in September to review the progress the world has made so far towards achieving the SDGs, at the midpoint of the 2030 Agenda, SDSN emphasizes six areas for immediate action.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investm

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_deberta
374,Although on average the world has made some pr...,"In response to feedback, since 2018 we have su...",0_15,0_30,NaN,No Relationship,No Relationship
891,"The interconnected environmental, social, and ...","The United States, as the world’s biggest econ...",13_3,13_5,NaN,Support,No Relationship
868,Today’s land-use practices and food systems ha...,"Yet the SDG Dashboards rate rich countries, in...",12_8,12_9,NaN,Support,No Relationship
935,"The GFA includes multilateral institutions, na...",47 countries have committed to submitting a VN...,16_3,16_7,NaN,No Relationship,No Relationship
405,Official high-level speeches and the preparati...,47 countries have submitted a VNR this year: o...,0_18,0_19,NaN,Support,No Relationship


rel_deberta
No Relationship    1046
Attack                6
Support               4
Rephrase              2
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 172

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: only limited progress is being made on the environmental and biodiversity goals, including SDG 12 (Responsible Consumption and Production), SDG 13 (Climate Action), SDG 14 (Life Below Water), and SDG 15 (Life on Land)
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: And global cooperation has ebbed as geopolitical tensions have risen.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical tha

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_deberta
4159,And global cooperation has ebbed as geopolitic...,Ensure healthy lives and promote well-being fo...,1_1,3_1,NaN,No Relationship,No Relationship
2874,The SDG Index serves as a conversation opener ...,Ensure sustainable consumption and production ...,0_28,12_2,NaN,No Relationship,No Relationship
9815,The world has made some progress in strengthen...,"To achieve this feat, countries have embarked ...",9_3,17_14,NaN,Support,No Relationship
2597,"At their core, the SDGs are an investment agen...","Provincial, metropolitan, and city governments...",0_0,12_5,NaN,No Relationship,No Relationship
1397,"By design, Transformation 5 calls for regional...",Zero-carbon energy systems,0_21,7_3,NaN,No Relationship,No Relationship


rel_deberta
No Relationship    11636
Attack                77
Rephrase              77
Support               32
Name: count, dtype: int64

#### Gemma3 27B extraction

In [6]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 201

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Nat

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_deberta
1764,"It aligns closely with SDG Target 3.8, achievi...","In the SDGs, universal health care (UHC) is co...",3_11,3_12,NaN,Support,No Relationship
1583,Questions explored policy measures to address ...,poor countries need help to combat poverty.,1_17,1_19,NaN,Support,No Relationship
3219,There is no hope for global peace unless there...,The Board’s 2023 report lists six areas for ac...,16_7,16_9,NaN,Support,No Relationship
2259,Global resource consumption assessments for ra...,Unsustainable consumption is strongly intercon...,12_0,12_13,NaN,Support,No Relationship
1570,Peace and global cooperation mean nothing less...,Disadvantaged communities have lower access to...,1_14,1_18,NaN,No Relationship,No Relationship


rel_deberta
No Relationship    4109
Attack               77
Support               1
Rephrase              1
Name: count, dtype: int64

In [7]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 253

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: As called for by United Nations Secretary-General António Guterres, the SDG Stimulus plan has five main components:
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (LICs) and lower-middle-income countries (LMICs), to carry out needed SDG actions;
  L

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_deberta
47795,UN Member States should commit to accelerating...,Countries promote global cooperation for susta...,15_1,16_24,NaN,Support,No Relationship
49985,Countries promote global cooperation for susta...,All UN Member States and UN agencies can count...,16_24,17_8,NaN,Support,No Relationship
13112,The SDGs are facing strong headwinds.,Achieving the SDGs requires global cooperation...,0_13,17_31,NaN,Support,No Relationship
8291,Achieving the SDGs requires global cooperation...,The adoption of the SDGs and the Paris Climate...,0_5,13_27,NaN,Support,No Relationship
40963,Meeting the SDGs and the Paris Agreement goals...,There continues to be a major discrepancy betw...,11_9,13_21,NaN,Attack,No Relationship


rel_deberta
No Relationship    49345
Attack               643
Rephrase              92
Support               17
Name: count, dtype: int64

#### Gemma3 4B extraction

In [8]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 268

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: None of their objectives are beyond our reach.
  Label: Attack

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The SDGs are still achievable.
  Label: Attack

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 203

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_deberta
19582,The SDGs are seriously off track (Figure 1.1)....,SDSN recognizes this gap in the practical mean...,11_1,11_11,NaN,No Relationship
2621,"UN Member States should adopt an SDG Stimulus,...",Extreme poverty rates in LICs remain above pre...,0_16,0_102,NaN,No Relationship
15840,HICs are able to mobilize vast financial resou...,"The goals related to hunger, sustainable diets...",3_19,3_26,NaN,No Relationship
3518,"Creation of ambitious, internationally-agreed ...",Governments must take the lead in all six area...,0_22,0_120,NaN,No Relationship
4416,they are grounded in the Universal Declaration...,"Unless the SDGs are actively pursued, geophysi...",0_29,0_38,NaN,No Relationship


rel_deberta
No Relationship    22638
Attack               262
Rephrase               9
Support                5
Name: count, dtype: int64

In [9]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 268

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Create 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_deberta
171612,globalized trade rules for ‘cleantech’ could a...,Revision of the credit-rating system and debt-...,7_13,9_5,NaN,No Relationship
142406,Governments are only now mapping out pathways ...,The capacity and healthy functioning of ecosys...,4_10,6_8,NaN,No Relationship
173067,They can also improve resource-use efficiencie...,Although all governments are in principle comm...,7_21,10_13,NaN,No Relationship
32609,Many EU member states demonstrate a high or mo...,global achievement of the SDGs rose only sligh...,0_53,7_2,NaN,No Relationship
5860,Each scorecard consists of a collection of hea...,Align private business investment flows with t...,0_122,1_4,NaN,No Relationship


rel_deberta
No Relationship    223413
Attack               1902
Rephrase              586
Support                50
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [10]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 237

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Since the outbreak of the pandemic in 2020 and other simultaneous crises, SDG progress has stalled globally.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_deberta
2572,Investing in the SDGs,The Nordic countries and European Union have s...,0_27,0_35,NaN,Support,No Relationship
952,"At the halfway mark to 2030, there remains a g...","Despite this ominous news, the SDGs are still ...",0_9,0_26,NaN,Support,No Relationship
11170,SDSN has joined the Group on Earth Observation...,SDG 17 (Partnerships for the Goals) calls for ...,17_26,17_47,NaN,Support,No Relationship
6060,Universal health access and coverage: an expan...,"In HICs and LICs, the pandemic and other crise...",3_0,3_10,NaN,No Relationship,No Relationship
9115,We emphasize that achieving the SDGs rests on ...,"Second, developed countries are not being held...",16_2,16_15,NaN,Attack,No Relationship


rel_deberta
No Relationship    11887
Attack               109
Support               13
Rephrase               2
Name: count, dtype: int64

In [11]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 200

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDG

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_deberta
8303,"The key finding from this survey is that, seve...","In Finland, 46 percent of parliamentarians are...",0_75,5_5,NaN,No Relationship,No Relationship
8987,We estimate that on average only around 18 per...,"According to IMF estimates in 2019, the financ...",0_60,6_5,NaN,Support,No Relationship
10826,We firmly believe that international cooperati...,Unemployment rates in both HICs and LICs is ab...,0_24,8_0,NaN,No Relationship,No Relationship
13335,"Strategically, the SDSN is very much committed...",In the 2030 Agenda for Sustainable Development...,0_99,9_19,NaN,No Relationship,No Relationship
5173,SDSN’s new flagship initiative – the SDG Trans...,"Those related to hunger, sustainable diets, an...",0_106,3_7,NaN,No Relationship,No Relationship


rel_deberta
No Relationship    17066
Attack                55
Rephrase               7
Support                4
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [12]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

Using MAX_LENGTH = 355

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable. None of their objectives are beyond our reach.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
-

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL,rel_llama,rel_deberta
14312,Revise liquidity structures for LICs and LMICs...,The SDSN is now a global network of more than ...,17_1,17_51,NaN,No Relationship,No Relationship,No Relationship
6041,"In July 2023, the EU is set to present its fir...",From a simple linear projection of past growth...,0_52,0_88,NaN,Support,Attack,No Relationship
1064,Achieving the SDGs requires global cooperation...,A large majority of governments – 83 percent o...,0_7,0_106,NaN,Support,Support,No Relationship
16617,It is also vital to share fairly and globally ...,One of the consistent findings of the SDSN is ...,17_45,17_46,NaN,Support,Support,No Relationship
6641,Even the progress on consistent national repor...,"Overall, HICs tend to generate the largest neg...",0_59,0_93,NaN,No Relationship,No Relationship,No Relationship


rel_deberta
No Relationship    16948
Attack               142
Support               19
Rephrase               4
Name: count, dtype: int64

In [13]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_deberta"].value_counts()

/tmp/ipykernel_19/2617973462.py:72: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Using MAX_LENGTH = 382

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv


/tmp/ipykernel_19/2617973462.py:100: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: 1. Increased funding from the multilateral develop-ment banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to inve

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL,rel_flanT5,rel_llama,rel_deberta
53786,"At the mid-point of the SDG agenda, we are far...","Working together with the IMF and the MDBs, th...",0_82,17_44,NaN,No Relationship,Support,NaN,No Relationship
81497,Governments are only now mapping out pathways ...,The SDSN is now a global network of more than ...,3_6,17_51,NaN,Support,Support,NaN,No Relationship
72972,The 2021 UN Food Systems Summit raised many ur...,Despite the world improving on average half a ...,2_1,15_14,NaN,Support,No Relationship,NaN,No Relationship
55223,The SDGs have a significant impact on public m...,We firmly believe that international cooperati...,0_101,17_18,NaN,Support,Support,NaN,No Relationship
24597,SDG transformations.,The report’s recommendations include calling f...,0_8,10_5,NaN,Support,Support,Support,No Relationship


rel_deberta
No Relationship    117438
Attack                655
Rephrase              111
Support                55
Name: count, dtype: int64